*Status: notes-only. Normalisation will start after cleaning.*

### Normalisation goals:
- Separate series name and number from title columns into own separate columns
- Separate table for publishers (one to many)
- Separate table for authors (+ junction many to many)
- Figure out how to tie sequels together - column for series name if applicable


**Assumption about authors:** 

- same name -> same entity 
_This will need to be verified in the future, so far this particular dataset doesn't provide unique identifier/differenciator for authors' names._
- means: author name will have UNIQUE constraint

**To check before splitting/confirming schema:**
- Figure out how many multiple-author series there are. Can series name be unique? How to identify series as distinct?
- Decide encoding on publication date

In [ ]:
from bookstats.config import CLEAN_DATA
from bookstats.formatting import clean_whitespace
import pandas as pd

df = pd.read_csv(CLEAN_DATA)

### Titles Containing Series

Reference: 
- Angels & Demons (Robert Langdon #1)
- A Killing Rain (Louis Kincaid #6)


In [ ]:
# Preview what comes up:

series = df[df["title"].str.contains(r" \([\w ]+ \#\d+\)", regex=True)]

series.value_counts()


In [ ]:
# Focus on most 'popular' ones and check those values:

series.sort_values("ratings_count").head(30)

In [ ]:
# Look at series with multiple authors listed

series_multiauthor = series[series["authors"].str.contains("/")]

series_multiauthor

In [ ]:
# Proportion of multi-author series to single-author (can I safely use author-name as composite key?)

print(len(series))
print(len(series_multiauthor))

percentage = len(series_multiauthor) / len(series) * 100
print(f"{round(percentage)} %")

Nearly third of series here are multi-authored. That means that I cannot have a single author + series name as my composite key that would guarantee correct uniqueness.

*Options*
- Surrogate PK, unique name column: Introduces risk of two genuinely different series of the same name colliding. 
- PK as composite of name + 1st listed author: Resolves doubling names, but may introduce series split purely on authors being listed in different order.

-> Because multi-author series present 27% of series data at the moment, it's safer to constrain name as unique.

**Series' authors conclusion:**

Because I'll procees with unique on name constraint, I am not including authors in series tables. The use case was only for differentiation, otherwise authors belong conceptually to specific books, not series. 

Authors are tied to series through books included in those series. You cannot 'author' a series without authoring a book.

### Extracting new series columns:

1. Series name + part numbering (series_name, series_part)

In [ ]:
df[["series_name", "series_part"]] = df["title"].str.extract(r" \(([\w ]+) \#(\d+)\)")

df

2. Split multiple authors within one string value

at first sight, authors seem to be grouped with "/" separator. Double check what comes up for other separators and symbols:

In [ ]:
# Take a look at how many rows have "/"" separators:

authors_slash = df[df["authors"].str.contains("/")]
authors_slash.count()

In [ ]:
total_count = len(df["authors"])
slash_count = len(authors_slash)

percentage = slash_count / total_count
print(f"{round(percentage * 100)} %")

Other delimiters:

In [ ]:
plus_mask = df["authors"].str.contains("+", regex=False)
comma_mask = df["authors"].str.contains(",", regex=False)
ampersand_mask = df["authors"].str.contains("&", regex=False)
and_mask = df["authors"].str.contains(" and ", regex=False)
semicolon_mask = df["authors"].str.contains(";", regex=False)
pipe_mask = df["authors"].str.contains("|", regex=False)
newline_mask = df["authors"].str.contains("\n", regex=False)

results = {
    "plus" : df[plus_mask]["authors"].nunique(),
    "comma" : df[comma_mask]["authors"].nunique(),
    "ampersand" : df[ampersand_mask]["authors"].nunique(),
    "and" : df[and_mask]["authors"].nunique(),
    "semicolon" : df[semicolon_mask]["authors"].nunique(),
    "pipe" : df[pipe_mask]["authors"].nunique(),
    "newline" : df[newline_mask]["authors"].nunique()
}

pd.Series(results)


In [ ]:
and_authors = df[and_mask]

and_authors

All author instances containing "and" are singular entity.

**Conclusion: "/" is the sole delimiter.**

### Split multiple authors into separate rows:

Author is defined here as entity inside "authors" column, where multiple are marked by "/".

In [ ]:
# Check behaviour:

df["authors_split"] = df["authors"].str.split("/")

df[["authors", "authors_split"]]

In [ ]:
df = df.explode("authors_split")
df["authors_split"] = clean_whitespace(df["authors_split"])

df[["authors", "authors_split"]]


### Future plans:
- Author entities get correctly categorised as authors, illustrators, translators etc. (many to many)
    -> Prepare a table of people-only (plus what they authored to differentiate people of the same name?), then use scraping/AI to find if they contributed as author/translator/illustrator
- People have ids, with profession it's probably many to many, since one person can be both an author and translator
- Scrape additional information about publishers, including location (which should dedup certain entities)